# Task 1: Video Preprocessing Pipeline

Pipeline tiền xử lý video cho rPPG multi-ROI: đọc frame, lấy metadata, phát hiện MediaPipe Face Mesh landmarks, trích xuất ROI trán/má trái/má phải, đánh dấu frame lỗi, và lưu artifact cho Task 2.

Notebook này được thiết kế để chạy với video local. Dataset không cần tồn tại trong repo.

## 1. Configuration and Dataset Connection

Edit the **User Configuration** block in the next cell, then run this same notebook. No environment variables are required.

Default behavior in this notebook is full-dataset preprocessing:

- `PROCESS_ALL_VIDEOS = True`
- `MAX_FRAMES = None` for full videos
- `BATCH_MAX_VIDEOS = None` for all discovered videos

For a quick debug run, temporarily set `MAX_FRAMES` or `BATCH_MAX_VIDEOS` in the config block.

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import tempfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
from tqdm.auto import tqdm


def path_has_non_ascii(path: Path) -> bool:
    try:
        str(path).encode("ascii")
    except UnicodeEncodeError:
        return True
    return False


def configure_mediapipe_resource_root() -> Path | None:
    """Work around MediaPipe graph loading on Windows when the repo path has Unicode.

    MediaPipe 0.10.x validates graph paths in native code; on some Windows
    setups it fails when site-packages lives under a non-ASCII project path.
    Copying MediaPipe resource files to the OS temp directory gives the graph
    loader an ASCII resource root while keeping the Python package unchanged.
    """
    package_root = Path(mp.__file__).resolve().parent
    graph_path = package_root / "modules" / "face_landmark" / "face_landmark_front_cpu.binarypb"
    if not graph_path.exists() or not path_has_non_ascii(package_root):
        return None

    import mediapipe.python.solution_base as solution_base

    target_root = Path(tempfile.gettempdir()) / "airppg_mediapipe_resources" / "mediapipe"
    target_modules = target_root / "modules"
    target_graph = target_modules / "face_landmark" / "face_landmark_front_cpu.binarypb"
    if not target_graph.exists():
        target_root.mkdir(parents=True, exist_ok=True)
        shutil.copytree(package_root / "modules", target_modules, dirs_exist_ok=True)
    (target_root / "python").mkdir(parents=True, exist_ok=True)
    solution_base.__file__ = str(target_root / "python" / "solution_base.py")
    return target_root


MEDIAPIPE_RESOURCE_ROOT = configure_mediapipe_resource_root()

VIDEO_EXTENSIONS = (".avi", ".mp4", ".mov", ".mkv")

# -----------------------------------------------------------------------------
# User Configuration
# -----------------------------------------------------------------------------
# Edit these values directly in the notebook. Do not hardcode private absolute
# paths before committing; keep dataset/output folders local and gitignored.

USER_DATASET_ROOT = Path("datasets/UBFC_DATASET")
USER_VIDEO_PATH: Path | None = None  # Example: Path("datasets/UBFC_DATASET/DATASET_2/subject1/vid.avi")
USER_OUTPUT_DIR = Path("outputs/task1_preprocessing")
USER_RUN_NAME: str | None = None

# Full-dataset mode. Keep True when you want to preprocess DATASET_1 and DATASET_2.
PROCESS_ALL_VIDEOS = True
BATCH_MAX_VIDEOS: int | None = None  # Example debug value: 2
SKIP_EXISTING = True

# Full video by default. Set an int only for debugging.
TARGET_FPS: float | None = None
RESIZE_WIDTH: int | None = 640
MAX_FRAMES: int | None = None  # Example debug value: 300
SAVE_OVERLAY_VIDEO = False
OVERLAY_SAMPLE_COUNT = 8
MIN_ROI_AREA_PX = 100.0
SCHEMA_VERSION = "task1_roi_v1"


def resolve_project_root() -> Path:
    """Return the repository root whether Jupyter starts in repo root or notebooks/."""
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "notebooks" else cwd



def discover_dataset_videos(dataset_root: Path) -> list[Path]:
    """Find video files in a local dataset folder without requiring a fixed UBFC layout."""
    if not dataset_root.exists():
        return []
    return sorted(
        path
        for path in dataset_root.rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )


def select_video_path(dataset_root: Path, videos: list[Path]) -> Path | None:
    """Use USER_VIDEO_PATH when set; otherwise pick the first discovered dataset video."""
    if USER_VIDEO_PATH is not None:
        configured = USER_VIDEO_PATH.expanduser()
        return configured.resolve() if configured.is_absolute() else (PROJECT_ROOT / configured).resolve()
    if videos:
        return videos[0]
    return None


def dataset_relative_parts(video_path: Path, dataset_root: Path) -> tuple[str, ...]:
    try:
        return video_path.relative_to(dataset_root).parts
    except ValueError:
        return video_path.parts[-3:]


def dataset_split_name(video_path: Path, dataset_root: Path) -> str:
    parts = dataset_relative_parts(video_path, dataset_root)
    return parts[0] if parts and parts[0].startswith("DATASET_") else "UNSORTED"


def video_subject_name(video_path: Path, dataset_root: Path) -> str:
    parts = dataset_relative_parts(video_path, dataset_root)
    if len(parts) >= 2 and parts[0].startswith("DATASET_"):
        return parts[1]
    return video_path.parent.name


def video_run_name(video_path: Path) -> str:
    return video_path.stem.replace(" ", "_")


def video_output_dir(video_path: Path, dataset_root: Path, output_root: Path) -> Path:
    return output_root / dataset_split_name(video_path, dataset_root) / video_subject_name(video_path, dataset_root)


def make_run_name(video_path: Path | None, dataset_root: Path) -> str:
    if USER_RUN_NAME is not None:
        return USER_RUN_NAME
    if video_path is None:
        return "sample_video"
    return video_run_name(video_path)


PROJECT_ROOT = resolve_project_root()
DATASET_ROOT = (PROJECT_ROOT / USER_DATASET_ROOT).resolve() if not USER_DATASET_ROOT.is_absolute() else USER_DATASET_ROOT.resolve()
DATASET_VIDEOS = discover_dataset_videos(DATASET_ROOT)
VIDEO_PATH = select_video_path(DATASET_ROOT, DATASET_VIDEOS)

OUTPUT_DIR = (PROJECT_ROOT / USER_OUTPUT_DIR).resolve() if not USER_OUTPUT_DIR.is_absolute() else USER_OUTPUT_DIR.resolve()
RUN_NAME = make_run_name(VIDEO_PATH, DATASET_ROOT)

CONFIG = {
    "target_fps": TARGET_FPS,
    "resize_width": RESIZE_WIDTH,
    "max_frames": MAX_FRAMES,
    "save_overlay_video": SAVE_OVERLAY_VIDEO,
    "overlay_sample_count": OVERLAY_SAMPLE_COUNT,
    "min_roi_area_px": MIN_ROI_AREA_PX,
    "schema_version": SCHEMA_VERSION,
}

BATCH_CONFIG = {
    "process_all_videos": PROCESS_ALL_VIDEOS,
    "batch_max_videos": BATCH_MAX_VIDEOS,
    "skip_existing": SKIP_EXISTING,
}

ROI_NAMES = ("forehead", "left_cheek", "right_cheek")
EXPECTED_LANDMARKS = 478  # MediaPipe Face Mesh with refine_landmarks=True.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATASET_ROOT = {DATASET_ROOT}")
print(f"Found {len(DATASET_VIDEOS)} video(s) under DATASET_ROOT")
for idx, path in enumerate(DATASET_VIDEOS[:10], start=1):
    print(f"  {idx:02d}. {path.relative_to(DATASET_ROOT)}")
if len(DATASET_VIDEOS) > 10:
    print(f"  ... {len(DATASET_VIDEOS) - 10} more")
print(f"VIDEO_PATH = {VIDEO_PATH if VIDEO_PATH is not None else '<unset>'}")
print(f"RUN_NAME = {RUN_NAME}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print("Dataset split counts:")
for split in sorted({dataset_split_name(path, DATASET_ROOT) for path in DATASET_VIDEOS}):
    count = sum(1 for path in DATASET_VIDEOS if dataset_split_name(path, DATASET_ROOT) == split)
    print(f"  {split}: {count} video(s)")
if MEDIAPIPE_RESOURCE_ROOT is not None:
    print(f"MEDIAPIPE_RESOURCE_ROOT = {MEDIAPIPE_RESOURCE_ROOT}")
print(f"CONFIG = {json.dumps(CONFIG, indent=2)}")
print(f"BATCH_CONFIG = {json.dumps(BATCH_CONFIG, indent=2)}")

## 2. Utility Functions

In [ ]:
@dataclass(frozen=True)
class VideoMetadata:
    path: str
    fps: float
    frame_count: int
    width: int
    height: int
    duration_seconds: float


def read_video_metadata(video_path: Path) -> VideoMetadata:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()

    duration = frame_count / fps if fps > 0 else 0.0
    return VideoMetadata(
        path=str(video_path),
        fps=fps,
        frame_count=frame_count,
        width=width,
        height=height,
        duration_seconds=duration,
    )


def resize_frame(frame_bgr: np.ndarray, resize_width: int | None) -> np.ndarray:
    if resize_width is None:
        return frame_bgr
    h, w = frame_bgr.shape[:2]
    if w == resize_width:
        return frame_bgr
    scale = resize_width / float(w)
    new_size = (resize_width, max(1, int(round(h * scale))))
    return cv2.resize(frame_bgr, new_size, interpolation=cv2.INTER_AREA)


def should_process_frame(frame_idx: int, source_fps: float, target_fps: float | None) -> bool:
    if target_fps is None or source_fps <= 0 or target_fps >= source_fps:
        return True
    source_time = frame_idx / source_fps
    previous_slot = math.floor(((frame_idx - 1) / source_fps) * target_fps) if frame_idx > 0 else -1
    current_slot = math.floor(source_time * target_fps)
    return current_slot > previous_slot


def landmarks_to_pixels(face_landmarks: Any, width: int, height: int) -> np.ndarray:
    points = np.full((EXPECTED_LANDMARKS, 2), np.nan, dtype=np.float32)
    for idx, landmark in enumerate(face_landmarks.landmark[:EXPECTED_LANDMARKS]):
        points[idx, 0] = landmark.x * width
        points[idx, 1] = landmark.y * height
    return points


def polygon_area(points: np.ndarray) -> float:
    if len(points) < 3 or np.isnan(points).any():
        return 0.0
    x = points[:, 0]
    y = points[:, 1]
    return float(0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def clip_polygon(points: np.ndarray, width: int, height: int) -> np.ndarray:
    clipped = points.copy()
    clipped[:, 0] = np.clip(clipped[:, 0], 0, width - 1)
    clipped[:, 1] = np.clip(clipped[:, 1], 0, height - 1)
    return clipped


def make_mask(shape_hw: tuple[int, int], polygon: np.ndarray) -> np.ndarray:
    mask = np.zeros(shape_hw, dtype=np.uint8)
    if len(polygon) >= 3 and not np.isnan(polygon).any():
        cv2.fillPoly(mask, [np.round(polygon).astype(np.int32)], 1)
    return mask


def write_image(path: Path, image_bgr: np.ndarray) -> None:
    """Write images reliably on Windows paths containing Unicode characters."""
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, encoded = cv2.imencode(path.suffix or ".png", image_bgr)
    if not ok:
        raise RuntimeError(f"Failed to encode image: {path}")
    encoded.tofile(str(path))


def read_image(path: Path) -> np.ndarray:
    """Read images reliably on Windows paths containing Unicode characters."""
    data = np.fromfile(str(path), dtype=np.uint8)
    image = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if image is None:
        raise RuntimeError(f"Failed to read image: {path}")
    return image

## 3. ROI Definition

The ROI polygons are defined from Face Mesh landmarks. These are practical starting regions for rPPG and should be reviewed visually on the target dataset.

In [ ]:
ROI_LANDMARKS = {
    # Upper face region above eyebrows. Avoids relying too much on the hairline.
    "forehead": [109, 10, 338, 337, 336, 296, 334, 293, 300, 70, 63, 105, 66, 107],
    # Subject left cheek.
    "left_cheek": [50, 101, 118, 119, 100, 47, 126, 209, 49, 203, 205, 187],
    # Subject right cheek.
    "right_cheek": [280, 330, 347, 348, 329, 277, 355, 429, 279, 423, 425, 411],
}
ROI_MAX_POINTS = max(len(indices) for indices in ROI_LANDMARKS.values())


def pack_roi_polygons(polygons: dict[str, np.ndarray]) -> np.ndarray:
    packed = np.full((len(ROI_NAMES), ROI_MAX_POINTS, 2), np.nan, dtype=np.float32)
    for roi_idx, roi_name in enumerate(ROI_NAMES):
        polygon = polygons[roi_name]
        packed[roi_idx, : len(polygon), :] = polygon
    return packed


def extract_roi_polygons(
    landmarks_px: np.ndarray,
    width: int,
    height: int,
    min_area_px: float,
) -> tuple[dict[str, np.ndarray], list[str]]:
    polygons: dict[str, np.ndarray] = {}
    errors: list[str] = []

    for roi_name in ROI_NAMES:
        idxs = ROI_LANDMARKS[roi_name]
        polygon = landmarks_px[idxs].astype(np.float32)
        polygon = clip_polygon(polygon, width=width, height=height)
        area = polygon_area(polygon)

        if area < min_area_px:
            errors.append(f"unstable_roi:{roi_name}")
            polygons[roi_name] = np.full((len(idxs), 2), np.nan, dtype=np.float32)
        else:
            polygons[roi_name] = polygon

    return polygons, errors


def draw_rois(frame_bgr: np.ndarray, polygons: dict[str, np.ndarray], valid: bool) -> np.ndarray:
    output = frame_bgr.copy()
    colors = {
        "forehead": (0, 220, 255),
        "left_cheek": (80, 220, 80),
        "right_cheek": (255, 160, 60),
    }

    for roi_name, polygon in polygons.items():
        if np.isnan(polygon).any():
            continue
        pts = np.round(polygon).astype(np.int32)
        cv2.polylines(output, [pts], isClosed=True, color=colors[roi_name], thickness=2)
        anchor = tuple(pts[0])
        cv2.putText(output, roi_name, anchor, cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[roi_name], 1)

    status = "valid" if valid else "invalid"
    cv2.putText(output, status, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0) if valid else (0, 0, 255), 2)
    return output

## 4. Main Pipeline

In [ ]:
def process_video(
    video_path: Path,
    output_dir: Path,
    run_name: str,
    config: dict[str, Any],
    progress_position: int = 0,
    leave_progress: bool = True,
) -> dict[str, Path]:
    metadata = read_video_metadata(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    output_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir = output_dir / f"{run_name}_overlay_samples"
    overlay_dir.mkdir(parents=True, exist_ok=True)

    landmarks_all: list[np.ndarray] = []
    roi_polygons_all: list[np.ndarray] = []
    roi_masks_all: list[np.ndarray] = []
    valid_all: list[bool] = []
    error_reasons: list[str] = []
    processed_frame_indices: list[int] = []
    overlay_samples: list[np.ndarray] = []

    video_writer = None
    face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    try:
        total = metadata.frame_count if config["max_frames"] is None else min(metadata.frame_count, config["max_frames"])
        valid_count = 0
        invalid_count = 0
        with tqdm(
            total=total,
            desc=f"Frames: {video_path.name}",
            unit="frame",
            dynamic_ncols=True,
            position=progress_position,
            leave=leave_progress,
        ) as progress:
            frame_idx = 0
            while True:
                ok, frame_bgr = cap.read()
                if not ok:
                    break
                if config["max_frames"] is not None and frame_idx >= config["max_frames"]:
                    break

                progress.update(1)
                if not should_process_frame(frame_idx, metadata.fps, config["target_fps"]):
                    frame_idx += 1
                    continue

                frame_bgr = resize_frame(frame_bgr, config["resize_width"])
                height, width = frame_bgr.shape[:2]
                processed_frame_indices.append(frame_idx)

                landmarks_px = np.full((EXPECTED_LANDMARKS, 2), np.nan, dtype=np.float32)
                polygons = {name: np.full((len(ROI_LANDMARKS[name]), 2), np.nan, dtype=np.float32) for name in ROI_NAMES}
                masks = np.zeros((len(ROI_NAMES), height, width), dtype=np.uint8)
                frame_errors: list[str] = []

                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                result = face_mesh.process(frame_rgb)
                if not result.multi_face_landmarks:
                    frame_errors.append("no_face")
                else:
                    landmarks_px = landmarks_to_pixels(result.multi_face_landmarks[0], width=width, height=height)
                    if np.isnan(landmarks_px).all():
                        frame_errors.append("invalid_landmarks")
                    else:
                        polygons, roi_errors = extract_roi_polygons(
                            landmarks_px,
                            width=width,
                            height=height,
                            min_area_px=float(config["min_roi_area_px"]),
                        )
                        frame_errors.extend(roi_errors)
                        for roi_idx, roi_name in enumerate(ROI_NAMES):
                            masks[roi_idx] = make_mask((height, width), polygons[roi_name])

                valid = len(frame_errors) == 0
                if valid:
                    valid_count += 1
                else:
                    invalid_count += 1
                error_reasons.append(";".join(frame_errors) if frame_errors else "")
                valid_all.append(valid)
                landmarks_all.append(landmarks_px)
                roi_polygons_all.append(pack_roi_polygons(polygons))
                roi_masks_all.append(masks)

                overlay = draw_rois(frame_bgr, polygons, valid=valid)
                if len(overlay_samples) < int(config["overlay_sample_count"]):
                    overlay_samples.append(overlay)

                if config["save_overlay_video"]:
                    if video_writer is None:
                        out_path = output_dir / f"{run_name}_roi_overlay.mp4"
                        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                        out_fps = config["target_fps"] or metadata.fps or 30.0
                        video_writer = cv2.VideoWriter(str(out_path), fourcc, float(out_fps), (width, height))
                    video_writer.write(overlay)

                if len(processed_frame_indices) == 1 or len(processed_frame_indices) % 10 == 0:
                    progress.set_postfix(
                        processed=len(processed_frame_indices),
                        valid=valid_count,
                        invalid=invalid_count,
                    )

                frame_idx += 1
    finally:
        face_mesh.close()
        cap.release()
        if video_writer is not None:
            video_writer.release()

    if not landmarks_all:
        raise RuntimeError("No frames were processed. Check VIDEO_PATH, max_frames, and target_fps.")

    sample_paths = []
    for sample_idx, overlay in enumerate(overlay_samples):
        path = overlay_dir / f"overlay_{sample_idx:03d}.png"
        write_image(path, overlay)
        sample_paths.append(path)

    arrays_path = output_dir / f"{run_name}_roi_data.npz"
    metadata_path = output_dir / f"{run_name}_metadata.json"

    np.savez_compressed(
        arrays_path,
        frame_indices=np.asarray(processed_frame_indices, dtype=np.int32),
        landmarks_px=np.stack(landmarks_all, axis=0),
        roi_polygons_px=np.stack(roi_polygons_all, axis=0),
        roi_masks=np.stack(roi_masks_all, axis=0),
        valid=np.asarray(valid_all, dtype=bool),
        error_reasons=np.asarray(error_reasons, dtype=object),
        roi_names=np.asarray(ROI_NAMES, dtype=object),
    )

    failure_counts = Counter(reason for reason in error_reasons if reason)
    metadata_json = {
        "schema_version": config["schema_version"],
        "video": metadata.__dict__,
        "config": config,
        "roi_names": list(ROI_NAMES),
        "roi_landmarks": ROI_LANDMARKS,
        "processed_frame_count": len(processed_frame_indices),
        "valid_frame_count": int(np.sum(valid_all)),
        "invalid_frame_count": int(len(valid_all) - np.sum(valid_all)),
        "failure_counts": dict(failure_counts),
        "overlay_samples": [str(path) for path in sample_paths],
    }
    metadata_path.write_text(json.dumps(metadata_json, indent=2), encoding="utf-8")
    print(f"Saved {len(sample_paths)} overlay sample(s) to: {overlay_dir}")
    print(f"Saved arrays to: {arrays_path}")
    print(f"Saved metadata to: {metadata_path}")

    return {"arrays": arrays_path, "metadata": metadata_path, "overlay_dir": overlay_dir}


def load_roi_artifact(npz_path: Path) -> dict[str, np.ndarray]:
    with np.load(npz_path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}

## 5. Run Pipeline

If `VIDEO_PATH` is not set, the notebook selects the first video discovered under `DATASET_ROOT`. The progress bar shows raw frames read, while the postfix shows how many frames were actually processed after FPS sampling plus valid/invalid counts.

In [ ]:
if BATCH_CONFIG["process_all_videos"]:
    print("Single-video run skipped because PROCESS_ALL_VIDEOS is True. Batch mode will process dataset videos below.")
elif VIDEO_PATH is None:
    print("No video found. Set USER_DATASET_ROOT to a dataset folder or USER_VIDEO_PATH to a video file.")
elif not VIDEO_PATH.exists():
    print(f"VIDEO_PATH does not exist: {VIDEO_PATH}")
elif not VIDEO_PATH.is_file():
    print(f"VIDEO_PATH is not a file: {VIDEO_PATH}")
else:
    print("Starting Task 1 preprocessing...")
    metadata_preview = read_video_metadata(VIDEO_PATH)
    print("Selected video metadata:")
    print(json.dumps(metadata_preview.__dict__, indent=2))
    single_output_dir = video_output_dir(VIDEO_PATH, DATASET_ROOT, OUTPUT_DIR)
    artifact_paths = process_video(VIDEO_PATH, single_output_dir, RUN_NAME, CONFIG)
    print("Task 1 preprocessing finished.")
    print("Saved artifacts:")
    print(json.dumps({key: str(value) for key, value in artifact_paths.items()}, indent=2))

## 6. Process Full Dataset

Set `PROCESS_ALL_VIDEOS = True` in the User Configuration block to preprocess every discovered video. Outputs mirror the dataset split and subject folders:

- `outputs/task1_preprocessing/DATASET_1/<subject-or-session>/`
- `outputs/task1_preprocessing/DATASET_2/<subject>/`

By default this processes full videos. Use `MAX_FRAMES` or `BATCH_MAX_VIDEOS` only for debugging.

In [ ]:
def process_dataset_videos(
    videos: list[Path],
    dataset_root: Path,
    output_root: Path,
    config: dict[str, Any],
    batch_config: dict[str, Any],
) -> list[dict[str, str]]:
    selected = videos
    max_videos = batch_config["batch_max_videos"]
    if max_videos is not None:
        selected = selected[: int(max_videos)]

    results: list[dict[str, str]] = []
    failures: list[dict[str, str]] = []
    print(f"Batch selected {len(selected)} / {len(videos)} video(s)")

    for video_path in tqdm(
        selected,
        desc="Dataset videos",
        unit="video",
        dynamic_ncols=True,
        position=0,
        leave=True,
    ):
        split = dataset_split_name(video_path, dataset_root)
        subject = video_subject_name(video_path, dataset_root)
        run_name = video_run_name(video_path)
        subject_output_dir = video_output_dir(video_path, dataset_root, output_root)
        arrays_path = subject_output_dir / f"{run_name}_roi_data.npz"
        metadata_path = subject_output_dir / f"{run_name}_metadata.json"

        if batch_config["skip_existing"] and arrays_path.exists() and metadata_path.exists():
            results.append({
                "split": split,
                "subject": subject,
                "video": str(video_path),
                "status": "skipped_existing",
                "arrays": str(arrays_path),
                "metadata": str(metadata_path),
            })
            continue

        try:
            artifact_paths = process_video(
                video_path,
                subject_output_dir,
                run_name,
                config,
                progress_position=1,
                leave_progress=False,
            )
            results.append({
                "split": split,
                "subject": subject,
                "video": str(video_path),
                "status": "processed",
                "arrays": str(artifact_paths["arrays"]),
                "metadata": str(artifact_paths["metadata"]),
                "overlay_dir": str(artifact_paths["overlay_dir"]),
            })
        except Exception as exc:
            failure = {
                "split": split,
                "subject": subject,
                "video": str(video_path),
                "status": "failed",
                "error": repr(exc),
            }
            failures.append(failure)
            results.append(failure)
            print(f"Failed {video_path}: {exc!r}")

    manifest_path = output_root / "task1_dataset_manifest.json"
    manifest = {
        "dataset_root": str(dataset_root),
        "output_root": str(output_root),
        "config": config,
        "batch_config": batch_config,
        "total_discovered_videos": len(videos),
        "selected_video_count": len(selected),
        "processed_count": sum(item["status"] == "processed" for item in results),
        "skipped_existing_count": sum(item["status"] == "skipped_existing" for item in results),
        "failed_count": len(failures),
        "results": results,
    }
    output_root.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Saved batch manifest to: {manifest_path}")
    print(json.dumps({
        "processed": manifest["processed_count"],
        "skipped_existing": manifest["skipped_existing_count"],
        "failed": manifest["failed_count"],
    }, indent=2))
    return results


if BATCH_CONFIG["process_all_videos"]:
    batch_results = process_dataset_videos(DATASET_VIDEOS, DATASET_ROOT, OUTPUT_DIR, CONFIG, BATCH_CONFIG)
else:
    print("Batch mode is disabled. Set PROCESS_ALL_VIDEOS = True to process the full dataset.")

## 7. Inspect Saved Artifact

In [ ]:
artifact_output_dir = video_output_dir(VIDEO_PATH, DATASET_ROOT, OUTPUT_DIR) if VIDEO_PATH is not None else OUTPUT_DIR
artifact_npz = artifact_output_dir / f"{RUN_NAME}_roi_data.npz"
artifact_json = artifact_output_dir / f"{RUN_NAME}_metadata.json"

if artifact_npz.exists() and artifact_json.exists():
    arrays = load_roi_artifact(artifact_npz)
    metadata = json.loads(artifact_json.read_text(encoding="utf-8"))
    print("NPZ arrays:")
    for key, value in arrays.items():
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    print("\nMetadata summary:")
    print(json.dumps({
        "processed_frame_count": metadata["processed_frame_count"],
        "valid_frame_count": metadata["valid_frame_count"],
        "invalid_frame_count": metadata["invalid_frame_count"],
        "failure_counts": metadata["failure_counts"],
    }, indent=2))
else:
    print("No artifact found yet. Run the pipeline cell with a valid VIDEO_PATH first.")

## 8. Display Overlay Samples

In [ ]:
artifact_output_dir = video_output_dir(VIDEO_PATH, DATASET_ROOT, OUTPUT_DIR) if VIDEO_PATH is not None else OUTPUT_DIR
overlay_dir = artifact_output_dir / f"{RUN_NAME}_overlay_samples"
sample_paths = sorted(overlay_dir.glob("*.png"))[: int(CONFIG["overlay_sample_count"])]

if not sample_paths:
    print(f"No overlay samples found yet in: {overlay_dir}")
else:
    cols = min(4, len(sample_paths))
    rows = math.ceil(len(sample_paths) / cols)
    plt.figure(figsize=(4 * cols, 3 * rows))
    for idx, path in enumerate(sample_paths, start=1):
        image_bgr = read_image(path)
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, idx)
        plt.imshow(image_rgb)
        plt.title(path.name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()